< from_template >

In [1]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template("금융 관련 {subject}에 대해 간단히 설명해주세요")
prompt_template.invoke({"subject": "주식투자"})


ChatPromptValue(messages=[HumanMessage(content='금융 관련 주식투자에 대해 간단히 설명해주세요', additional_kwargs={}, response_metadata={})])

< from_messages >

In [2]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 금융 전문가입니다."),
    ("user", "{subject}에 대해 간단히 설명해주세요")
])
# '주식' 주제로 챗 프롬프트 템플릿 호출
prompt_template.invoke({"subject": "주식투자"})

ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 전문가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='주식투자에 대해 간단히 설명해주세요', additional_kwargs={}, response_metadata={})])

< MessagesPlaceholder >

In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

# (방법1) 메시지 자리 표시자를 포함한 챗 프롬프트 템플릿 정의
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "당신은 금융 전문가입니다."),
    MessagesPlaceholder("msgs")
])

prompt_template.invoke({"msgs": [HumanMessage(content="안녕하세요!")]})


ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 조언가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하세요!', additional_kwargs={}, response_metadata={})])

In [4]:
# (방법2) MessagesPlaceholder 클래스를 명시적으로 사용하지 않고도 비슷한 작업을 수행할 수 있는 방법
prompt_template = ChatPromptTemplate.from_messages([
("system", "당신은 금융 전문가입니다."),
("placeholder", "{msgs}") # <-- 여기서 'msgs'가 MessagesPlaceholder 대신 사용됩니다.
])

prompt_template.invoke({"msgs": [HumanMessage(content="안녕하세요!")]})


ChatPromptValue(messages=[SystemMessage(content='당신은 유능한 금융 조언가입니다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하세요!', additional_kwargs={}, response_metadata={})])

< PromptTemplate 를 이용한 few-shot 프롬프트>

In [7]:
from langchain_core.prompts import PromptTemplate
# 질문과 답변을 포맷하는 프롬프트 템플릿 정의
example_prompt = PromptTemplate.from_template("질문: {question}\n답변: {answer}")

examples = [
    {
        "question": "놀라다",
        "answer": "침착하다",
    } ,
    {
        "question": "만족하다",
        "answer": "불만족하다",
    },
    {
        "question": "춥다",
        "answer": "덥다",
    },
]

print(example_prompt.invoke(examples[0]).to_string())

질문: 놀라다
답변: 침착하다


< FewShotPromptTemplate 를 이용한 퓨샷 프롬프트 >

In [8]:
from langchain_core.prompts import FewShotPromptTemplate
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="질문: {input}",
    input_variables=["input"],
)

print(
    prompt.invoke({"input": "기쁘다"}).to_string()
)

질문: 놀라다
답변: 침착하다

질문: 만족하다
답변: 불만족하다

질문: 춥다
답변: 덥다

질문: 기쁘다


In [10]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_openai import ChatOpenAI

from dotenv import load_dotenv

load_dotenv("../../.env")

# 예제 프롬프트 템플릿 생성
example_prompt = PromptTemplate(
    input_variables=["question", "answer"],
    template="질문: {question}\n답변: {answer}"
)

# 퓨샷 프롬프트 템플릿 설정
prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    prefix="다음은 금융 관련 질문과 답변의 예시입니다:",
    suffix="질문: {input}",
    input_variables=["input"]
)

model = ChatOpenAI(model="gpt-4o-mini")

chain = prompt | model

response = chain.invoke({"input": "기쁘다."})  # invoke 메서드 사용
print(response.content)

답변: 슬프다
